In [1]:
import pandas as pd
from datasets import load_dataset

/scratch/s3799042/venvs/think-reduction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/scratch/s3799042/venvs/think-reduction/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
dataset_gsm8k = load_dataset("openai/gsm8k", "main", split="test")
df_gsm8k = pd.DataFrame(dataset_gsm8k)
print(f"GSM8K: {len(df_gsm8k)} problems")
print(f"Columns: {list(df_gsm8k.columns)}")
df_gsm8k.head()


GSM8K: 1319 problems
Columns: ['question', 'answer']


,question,answer
0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...
1,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...
2,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...
3,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...
4,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t..."


In [3]:
dataset_olympiad = load_dataset("math-ai/olympiadbench") # option test_en, test_cn
df_olympiad = pd.DataFrame(dataset_olympiad["test"])
print(f"Olympiad: {len(df_olympiad)} problems")
print(f"Columns: {list(df_olympiad.columns)}")
df_olympiad.head()


Olympiad: 674 problems
Columns: ['id', 'question', 'solution', 'final_answer', 'context', 'image_1', 'image_2', 'image_3', 'image_4', 'image_5', 'modality', 'difficulty', 'is_multiple_answer', 'unit', 'answer_type', 'error', 'question_type', 'subfield', 'subject', 'language']


,id,question,solution,final_answer,context,image_1,image_2,image_3,image_4,image_5,modality,difficulty,is_multiple_answer,unit,answer_type,error,question_type,subfield,subject,language
0,1606,Xenia and Sergey play the following game. Xeni...,[Sergey can determine Xenia's number in 2 but ...,[2],None,None,None,None,None,None,Text-only,Competition,False,None,Numerical,None,Open-ended,Combinatorics,Math,English
1,1610,"Given a positive integer $n$, determine the la...",[The required maximum is $\frac{1}{2 n+2}$. To...,[$\frac{1}{2 n+2}$],None,None,None,None,None,None,Text-only,Competition,False,None,Expression,None,Open-ended,Algebra,Math,English
2,1612,\nFind (in closed form) the difference between...,"[For every integer $M \geq 0$, let $A_{M}=\sum...",[$2^{1009}$],None,None,None,None,None,None,Text-only,Competition,False,None,Numerical,None,Open-ended,Number Theory,Math,English
3,1613,Determine all positive integers $n$ satisfying...,"[There is only one such integer, namely, $n=2$...",[2],None,None,None,None,None,None,Text-only,Competition,False,None,Numerical,None,Open-ended,Algebra,Math,English
4,1614,Let $n$ be an integer greater than 1 and let $...,[The required maximum is $2 n-2$. To describe ...,[$2n-2$],None,None,None,None,None,None,Text-only,Competition,False,None,Expression,None,Open-ended,Combinatorics,Math,English


In [4]:
dataset_amc = load_dataset("math-ai/amc23")
df_amc = pd.DataFrame(dataset_amc['test'])
print(f"AMC: {len(df_amc)} problems")
print(f"Columns: {list(df_amc.columns)}")
df_amc.head()


AMC: 40 problems
Columns: ['id', 'question', 'answer', 'url']


,id,question,answer,url
0,0,Cities $A$ and $B$ are $45$ miles apart. Alici...,27,https://artofproblemsolving.com/wiki/index.php...
1,1,Positive real numbers $x$ and $y$ satisfy $y^3...,36,https://artofproblemsolving.com/wiki/index.php...
2,2,What is the degree measure of the acute angle ...,45,https://artofproblemsolving.com/wiki/index.php...
3,3,What is the value of\n\[2^3 - 1^3 + 4^3 - 3^3 ...,3159,https://artofproblemsolving.com/wiki/index.php...
4,4,In a table tennis tournament every participant...,36,https://artofproblemsolving.com/wiki/index.php...


In [5]:
import pandas as pd

# Target schema
df_all = pd.DataFrame(columns=["dataset", "prompt", "solution_col", "level"])

dfs = []

df_math_aime = []


# GSM8K
dfs.append(pd.DataFrame({
    "dataset": "gsm8k",
    "prompt": df_gsm8k["question"],
    "solution_col": df_gsm8k["answer"],
    "level": None
}))

# Olympiad
dfs.append(pd.DataFrame({
    "dataset": "olympiad",
    "prompt": df_olympiad["question"],
    "solution_col": df_olympiad["final_answer"],  
    "level": None
}))

# AMC
dfs.append(pd.DataFrame({
    "dataset": "amc",
    "prompt": df_amc["question"],
    "solution_col": df_amc["answer"],
    "level": None
}))


df_all = pd.concat(dfs, ignore_index=True)






In [6]:
import re
def extract_boxed(s: str) -> str | None:
    # GSM8K
    matches = re.findall(
        r"(?m)^[ \t]*####[ \t]*([^\n\r#]+?)[ \t]*$",
        s
    )
    if matches:
        return matches[-1].strip()
    # Olympiad
    m = re.search(r"\$([^$]*)\$", s)
    if m:
        return m.group(1).strip()
    # AMC 
    m = re.search(r"(?m)^[ \t]*([+-]?\d+(?:\.\d+)?)[ \t]*$", s)
    if m:
        return m.group(1)
    
    return s



In [7]:
df_all = df_all.reset_index(drop=True)
df_all["id"] = df_all.index
df_all = df_all[["id", "dataset", "prompt", "solution_col", "level"]]
print(f"Final combined dataframe: {len(df_all)} problems")
df_all.tail(5)

Final combined dataframe: 2033 problems


,id,dataset,prompt,solution_col,level
2028,2028,amc,You are playing a game. A $2 \times 1$ rectang...,4,None
2029,2029,amc,When the roots of the polynomial \n\[P(x) = (...,6,None
2030,2030,amc,For how many integers $n$ does the expression\...,901,None
2031,2031,amc,"How many nonempty subsets $B$ of ${0, 1, 2, 3,...",144,None
2032,2032,amc,What is the area of the region in the coordina...,8,None


In [8]:
def normalize_solution(x):
    if isinstance(x, list):
        # join list elements into a single string
        return "\n".join(map(str, x))
    elif x is None:
        return None
    else:
        return str(x)

def normalize_level(x):
    """Remove "Level " prefix from level strings.
       Cast float levels to int
       set nan levels to None
    """
    if isinstance(x, str) and x.startswith("Level "):
        x = x.replace("Level ", "")
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    elif pd.isna(x):
        return None
    else:
        return str(x)

def deduplicate_dataframe(df):
    """Deduplicate dataframe based on 'problem' column."""
    before_dedup = len(df)
    df_dedup = df.drop_duplicates(subset=["prompt"])
    after_dedup = len(df_dedup)
    print(f"Removed {before_dedup - after_dedup} duplicate problems. New size: {after_dedup}")
    return df_dedup

In [9]:
df_all["solution_col"] = df_all["solution_col"].apply(normalize_solution)
df_all["level"] = df_all["level"].apply(normalize_level)
df_all = deduplicate_dataframe(df_all)
#reset id column to be sequential



Removed 0 duplicate problems. New size: 2033


In [10]:
#drop entries with level = "?"
df_all = df_all.drop(df_all[df_all["level"] == "?"].index)

In [11]:
df_all = df_all.reset_index(drop=True)
df_all["id"] = df_all.index

In [12]:
cols_with_none = df_all[df_all['solution_col'].isna()]
cols_with_none

,id,dataset,prompt,solution_col,level


In [13]:
solutions = []
for i in range(len(df_all)):
    row = df_all.iloc[i]
    solution = extract_boxed(row['solution_col'])
    solutions.append(solution)

In [14]:
df_cleaned = df_all
df_cleaned['solution_col'] = solutions
df_cleaned

,id,dataset,prompt,solution_col,level
0,0,gsm8k,Janet’s ducks lay 16 eggs per day. She eats th...,18,None
1,1,gsm8k,A robe takes 2 bolts of blue fiber and half th...,3,None
2,2,gsm8k,Josh decides to try flipping a house. He buys...,70000,None
3,3,gsm8k,James decides to run 3 sprints 3 times a week....,540,None
4,4,gsm8k,"Every day, Wendi feeds each of her chickens th...",20,None
...,...,...,...,...,...
2028,2028,amc,You are playing a game. A $2 \times 1$ rectang...,4,None
2029,2029,amc,When the roots of the polynomial \n\[P(x) = (...,6,None
2030,2030,amc,For how many integers $n$ does the expression\...,901,None
2031,2031,amc,"How many nonempty subsets $B$ of ${0, 1, 2, 3,...",144,None


In [16]:
from pathlib import Path
df_all["dataset"].unique()

df_all.to_parquet(Path().resolve().parent.parent / "processed" / "eval_L1_bert" / "gsm8k_olympiad_amc.parquet")